# Mục 5.4 — Thực hành phân cụm

Sinh tu ban thao giao trinh. Chay tuan tu tu tren xuong; moi o phu thuoc cac o truoc no.


## 5.4. Thực hành

Mục này áp dụng các thuật toán và chỉ số đã học vào một bài thực hành tổng hợp bằng Python (scikit-learn, numpy, pandas, matplotlib): nhiều thuật toán trên cùng một tập dữ liệu và so sánh kết quả.

### Chuẩn bị môi trường và tập dữ liệu

### Yêu cầu môi trường

Dùng môi trường thống nhất ở Phụ lục B: Python 3.11 hoặc 3.12 và scikit-learn 1.8.x. Cài các thư viện trong môi trường ảo bằng lệnh:

**Kết quả đối chiếu**

```
pip install numpy pandas scikit-learn matplotlib seaborn
```

### Tập dữ liệu sử dụng

IRIS: Bài thực hành dùng tập dữ liệu Iris kinh điển có sẵn trong scikit-learn. Tập dữ liệu gồm 150 mẫu hoa thuộc 3 loài (Setosa, Versicolor, Virginica) với 4 đặc trưng: chiều dài đài hoa, chiều rộng đài hoa, chiều dài cánh hoa, chiều rộng cánh hoa.

Lưu ý: Iris có sẵn nhãn loài, nhưng trong phân cụm, nhãn này được tạm coi là chưa biết. Nhãn chỉ được dùng để kiểm tra cuối cùng xem các thuật toán có tìm được đúng 3 nhóm tự nhiên hay không.

Đây là tập dữ liệu điển hình để thực hành phân cụm vì có cấu trúc rõ (3 cụm tự nhiên), kích thước vừa phải, và đặc trưng số học dễ xử lý.



In [ ]:
# Buoc 1: Load dataset Iris
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X = iris.data       # 150 mau, 4 dac trung
y = iris.target     # nhan that, KHONG dung cho phan cum
feature_names = iris.feature_names

print('Kich thuoc:', X.shape)
print('Dac trung:', feature_names)
print('So loai hoa:', len(np.unique(y)))


### Tiền xử lý và chuẩn hóa dữ liệu

### Vì sao cần chuẩn hóa

Bốn đặc trưng của Iris đều đo bằng cm nhưng phạm vi khác nhau (chiều dài đài hoa 4-8 cm, chiều rộng cánh hoa 0,1-2,5 cm); nếu không chuẩn hóa, các thuật toán dùng khoảng cách sẽ ưu tiên đặc trưng có phạm vi lớn. StandardScaler (Mục 2.4) đưa các đặc trưng về trung bình 0, độ lệch chuẩn 1.



In [ ]:
# Buoc 2: Chuan hoa du lieu
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Trung binh sau chuan hoa:', X_scaled.mean(axis=0))
print('Do lech chuan sau chuan hoa:', X_scaled.std(axis=0))
# Ket qua mong doi: trung binh xap xi 0, do lech chuan xap xi 1


### Bài thực hành 1: Áp dụng K-means

### Mục tiêu

Áp dụng K-means với k = 3 (đã biết Iris có 3 loài). Đánh giá kết quả bằng Silhouette và Davies-Bouldin. Mở rộng: thử k từ 2 đến 6, vẽ biểu đồ Silhouette và DBI theo k để chọn k tối ưu (tạm coi như chưa biết k = 3).



In [ ]:
# Buoc 3.1: Ap dung K-means voi k=3
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

kmeans = KMeans(n_clusters=3, init='k-means++',
                n_init=10, random_state=42)
labels_kmeans = kmeans.fit_predict(X_scaled)

sil_km = silhouette_score(X_scaled, labels_kmeans)
dbi_km = davies_bouldin_score(X_scaled, labels_kmeans)
print(f'K-means k=3: Silhouette={sil_km:.3f}, DBI={dbi_km:.3f}')

# Buoc 3.2: Tim k toi uu
import matplotlib.pyplot as plt

ks = range(2, 7)
sils, dbis = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    lbl = km.fit_predict(X_scaled)
    sils.append(silhouette_score(X_scaled, lbl))
    dbis.append(davies_bouldin_score(X_scaled, lbl))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(list(ks), sils, marker='o')
ax[0].set_xlabel('So cum k'); ax[0].set_ylabel('Silhouette')
ax[0].set_title('Silhouette theo k')
ax[1].plot(list(ks), dbis, marker='o', color='red')
ax[1].set_xlabel('So cum k'); ax[1].set_ylabel('DBI')
ax[1].set_title('Davies-Bouldin theo k')
plt.tight_layout(); plt.show()


Hình 5.11. Silhouette và DBI theo k trên Iris

### Kết quả mong đợi

Kết quả chạy lại với scikit-learn 1.8.0, random_state = 42, n_init = 10: Silhouette cao nhất tại k = 2 (0,582), tại k = 3 là 0,460. DBI tương ứng là 0,593 và 0,834; DBI thấp nhất tại k = 2 trong các giá trị 2–6 đã thử. Hình 5.11 đặt hai đường cạnh nhau cho thấy hai chỉ số cùng chỉ về một kết luận: Silhouette giảm đều từ k = 2 trở đi còn DBI tăng đều, nên cả hai đều chọn k = 2 chứ không phải k = 3 như số loài hoa.

Theo hai chỉ số này, k = 2 được ưu tiên trong phạm vi thử nghiệm. Nếu mục tiêu là đối chiếu ba loài hoa, vẫn có thể khảo sát k = 3; số lớp sinh học không bảo đảm bằng số cụm được nhận diện từ các đặc trưng và độ đo đã chọn.

### Bài thực hành 2: So sánh nhiều thuật toán

### Mục tiêu

Áp dụng các thuật toán đã học (K-means, K-medoids, FCM, DBSCAN, GMM, Agglomerative) trên cùng tập dữ liệu Iris đã chuẩn hóa. So sánh kết quả qua Silhouette và DBI.

Lưu ý: K-medoids và FCM cần cài đặt thư viện ngoài (kmedoids cho K-medoids, như Mục 5.2.1, scikit-fuzzy hoặc tự cài đặt cho FCM). Trong bài này tập trung các thuật toán có sẵn trong scikit-learn.



In [ ]:
# Buoc 4: So sanh cac thuat toan
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

results = {}

# K-means
km = KMeans(n_clusters=3, n_init=10, random_state=42)
results['K-means'] = km.fit_predict(X_scaled)

# Agglomerative voi Ward linkage
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
results['Agglomerative'] = agg.fit_predict(X_scaled)

# GMM
gmm = GaussianMixture(n_components=3, random_state=42)
results['GMM'] = gmm.fit_predict(X_scaled)

# DBSCAN (khong can so cum)
db = DBSCAN(eps=0.8, min_samples=5)
results['DBSCAN'] = db.fit_predict(X_scaled)

# Tinh Silhouette va DBI cho moi thuat toan
print(f'{"Thuat toan":<15} {"Silhouette":<12} {"DBI":<10}')
for name, labels in results.items():
    # Loai bo diem nhieu (label = -1) khi tinh chi so
    mask = labels != -1
    n_clusters = len(np.unique(labels[mask]))
    print(name, "So cum:", n_clusters, "So nhieu:", (~mask).sum())
    if 2 <= n_clusters < mask.sum():
        sil = silhouette_score(X_scaled[mask], labels[mask])
        dbi = davies_bouldin_score(X_scaled[mask], labels[mask])
        print(f'{name:<15} {sil:<12.3f} {dbi:<10.3f}')


### Phân tích kết quả

Kết quả chạy lại với scikit-learn 1.8.0 và cấu hình trong mã:

K-means: Silhouette = 0,460;

DBI = 0,834.

- Agglomerative (Ward): Silhouette = 0,447; DBI = 0,803.

- GMM: Silhouette = 0,475; DBI = 0,887.

- DBSCAN với eps = 0,8, min_samples = 5: Silhouette = 0,598; DBI = 0,569 sau khi bỏ điểm nhiễu khỏi phép tính.

K-means, Ward và GMM có kết quả khác nhau giữa hai chỉ số. DBSCAN được đánh giá sau khi bỏ nhiễu nên không trực tiếp dùng điểm số cao hơn để kết luận tốt hơn: cần báo cáo số cụm, số điểm bị loại và tỷ lệ dữ liệu được giữ. So sánh trên cùng tập điểm hoặc giải thích rõ khác biệt về phạm vi đánh giá.

### Bài thực hành 3: Trực quan hóa kết quả

### Vấn đề với dữ liệu 4 chiều

Iris có 4 đặc trưng, không thể vẽ trực tiếp lên biểu đồ 2D. Cần rút gọn chiều dữ liệu xuống 2D để trực quan hóa. Cách phổ biến: dùng PCA (Phân tích thành phần chính) đã giới thiệu ở Mục 2.5 hoặc t-SNE để giảm xuống 2 chiều. PCA chiếu dữ liệu lên hai trục có phương sai lớn nhất.



In [ ]:
# Buoc 5: Truc quan hoa bang PCA
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
algorithms = ['K-means', 'Agglomerative', 'GMM', 'DBSCAN']

for ax, name in zip(axes.flat, algorithms):
    labels = results[name]
    scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1],
                         c=labels, cmap='viridis',
                         edgecolor='black', s=50)
    ax.set_title(f'{name}')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

plt.tight_layout(); plt.show()


Hình 5.12. So sánh 4 thuật toán trên Iris (chiếu PCA 2D)

### Diễn giải hình ảnh

Trên cấu hình minh họa, nhóm Setosa tách tương đối rõ, còn Versicolor và Virginica có vùng chồng lấn. DBSCAN có thể gộp hai nhóm sau tùy eps và min_samples. Kết quả của một cấu hình không đủ để kết luận thuật toán luôn không phù hợp; cần thử tham số và kiểm tra cấu trúc dữ liệu. Hình 5.12 cho thấy rõ điều đó: ba thuật toán cần biết trước k đều chia vùng chồng lấn thành hai cụm nhưng ở các vị trí ranh giới khác nhau, còn DBSCAN với eps = 0,8 gộp cả vùng chồng lấn thành một cụm và đẩy bốn điểm ra ngoài thành nhiễu.

### Bài thực hành 4: Vẽ dendrogram cho Agglomerative

### Mục tiêu

Vẽ dendrogram cho tập dữ liệu Iris đã chuẩn hóa. Quan sát cấu trúc phân cấp và chọn số cụm bằng cách cắt dendrogram.



In [ ]:
# Buoc 6: Ve dendrogram
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Tinh ma tran ghep voi Ward linkage
Z = linkage(X_scaled, method='ward')

plt.figure(figsize=(14, 6))
dendrogram(Z, truncate_mode='lastp', p=30,
           leaf_rotation=90, leaf_font_size=10,
           show_contracted=True)
plt.title('Dendrogram cho Iris (Ward linkage)')
plt.xlabel('Mau du lieu (hoac cum)')
plt.ylabel('Khoang cach gop')
plt.axhline(y=10, color='red', linestyle='--',
            label='Cat o do cao 10 -> 3 cum')
plt.legend(); plt.show()

# Cat dendrogram de duoc 3 cum
clusters = fcluster(Z, t=3, criterion='maxclust')
print('Nhan cum tu dendrogram:', clusters[:20])


### Ý nghĩa dendrogram

Dendrogram của Iris (Ward linkage) thường cho thấy hai mức gộp lớn rõ rệt: một mức tách Setosa khỏi {Versicolor, Virginica} và một mức tách Versicolor và Virginica. Khoảng nhảy giữa hai mức cuối cùng lớn cho thấy có thể chọn k = 2 (Setosa vs phần còn lại) hoặc k = 3 (3 loài riêng biệt) tùy mục tiêu. Đây là một trong những ưu điểm của phân cụm phân cấp: cho phép người dùng linh hoạt chọn số cụm dựa trên hiểu biết về bài toán.

### Bài thực hành mở rộng: tập dữ liệu tự chọn

### Gợi ý các tập dữ liệu

Có thể thử Wine (178 mẫu, 13 đặc trưng, 3 lớp), Breast Cancer Wisconsin Diagnostic (569 mẫu, 30 đặc trưng, 2 lớp), Digits (ảnh chữ số 8 × 8) qua sklearn.datasets; hoặc dữ liệu khách hàng, sinh viên được phép sử dụng. Nhãn có sẵn chỉ dùng để đối chiếu sau phân cụm, không đưa vào đặc trưng đầu vào.

### Quy trình chuẩn

Áp dụng quy trình 6 bước đã học ở Mục 5.1.2 cho mỗi tập dữ liệu mới:

- (1) Xác định mục tiêu phân cụm

- (2) Chọn đặc trưng và tiền xử lý

- (3) Chọn thuật toán phù hợp

- (4) Chọn k và tham số

- (5) Thực hiện phân cụm

- (6) Đánh giá và diễn giải. Báo cáo nên có: mô tả tập dữ liệu, các bước thực hiện, biểu đồ trực quan, bảng so sánh chỉ số, và diễn giải ý nghĩa kinh doanh hoặc khoa học của các cụm tìm được

### Những lưu ý quan trọng khi thực hành

### Lưu ý 1: tính ngẫu nhiên

K-means và GMM có thành phần ngẫu nhiên (khởi tạo). Để kết quả tái lặp được, luôn đặt random_state=42 hoặc một số cố định. Khi báo cáo kết quả, nếu có thể, chạy thuật toán nhiều lần với các random_state khác nhau và lấy trung bình các chỉ số. Điều này giúp đánh giá đáng tin cậy hơn.

### Lưu ý 2: chuẩn hóa hay không

Chuẩn hóa thường cải thiện kết quả khi các đặc trưng có thang đo khác nhau. Tuy nhiên, có vài trường hợp không cần:

- (a) Tất cả đặc trưng đã có cùng thang đo (ví dụ tất cả đều là tỷ lệ phần trăm)

- (b) Một số đặc trưng quan trọng hơn các đặc trưng khác và cần giữ tỉ trọng (ít gặp)

- (c) Độ đo và trọng số đã được thiết kế theo ý nghĩa nghiệp vụ. DBSCAN cũng phụ thuộc độ đo lân cận, nên không mặc nhiên bỏ qua chuẩn hóa.

### Lưu ý 3: không có thuật toán hoàn hảo

Không có thuật toán nào tốt nhất cho mọi tập dữ liệu. Bảng tóm tắt dưới đây giúp chọn thuật toán phù hợp:

Bảng 5.15. Hướng dẫn chọn thuật toán phân cụm

### Lưu ý 4: giải thích kết quả

Sau khi có kết quả phân cụm, bước quan trọng là giải thích ý nghĩa các cụm. Một số cách:

- (a) Tính giá trị trung bình từng đặc trưng cho mỗi cụm. So sánh để hiểu đặc tính nổi bật của mỗi cụm

- (b) Đặt tên gợi nhớ cho mỗi cụm dựa trên đặc tính (ví dụ 'khách hàng VIP', 'sinh viên có nguy cơ')

- (c) Vẽ biểu đồ phân bố các đặc trưng theo cụm để trực quan

- (d) Trình bày kết quả với đối tượng không chuyên: dùng biểu đồ trực quan và ngôn ngữ đơn giản, tránh thuật ngữ chuyên môn

